# Student Performance Analysis & Prediction

**Slab 1 – For Beginners**

This notebook is designed as a complete, reproducible submission. Run the cells from top to bottom.

## 1. Objective
Analyze student performance and build regression models to predict the final grade (**G3**). The analysis examines study time, attendance/absences, previous grades, parental education, internet access and extracurricular activities.

**Dataset:** UCI Student Performance dataset. It contains Mathematics (`student-mat.csv`) and Portuguese (`student-por.csv`) versions.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

sns.set_theme(style="whitegrid")

# UCI public dataset ZIP
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"

# Download and read the math file directly from the ZIP.
import io, zipfile, requests
content = requests.get(DATA_URL, timeout=60).content
with zipfile.ZipFile(io.BytesIO(content)) as z:
    df = pd.read_csv(z.open("student-mat.csv"), sep=";")

print("Shape:", df.shape)
display(df.head())

## 2. Cleaning and exploratory analysis

In [ ]:
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).head(15))
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Final grade summary:")
display(df["G3"].describe().to_frame().T)

plt.figure(figsize=(9,5))
sns.histplot(df["G3"], bins=15, kde=True)
plt.title("Distribution of Final Grade (G3)")
plt.xlabel("Final Grade")
plt.show()

plt.figure(figsize=(9,5))
sns.boxplot(data=df, x="studytime", y="G3")
plt.title("Study Time vs Final Grade")
plt.show()

plt.figure(figsize=(9,5))
sns.scatterplot(data=df, x="absences", y="G3", alpha=0.65)
plt.title("Absences vs Final Grade")
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(df.select_dtypes(include=np.number).corr(), cmap="coolwarm", center=0)
plt.title("Numerical Correlation Heatmap")
plt.show()

## 3. Analyze requested factors

In [ ]:
factor_tables = {
    "Study time": df.groupby("studytime")["G3"].agg(["count","mean","median"]),
    "Internet access": df.groupby("internet")["G3"].agg(["count","mean","median"]),
    "Extracurricular activities": df.groupby("activities")["G3"].agg(["count","mean","median"]),
    "Mother education": df.groupby("Medu")["G3"].agg(["count","mean","median"]),
    "Father education": df.groupby("Fedu")["G3"].agg(["count","mean","median"]),
}
for name, table in factor_tables.items():
    print("\n", name)
    display(table.round(2))

# Previous scores are especially important in this dataset.
display(df[["G1","G2","G3"]].corr().round(3))

## 4. Machine-learning preparation
We predict G3 as a continuous score. Two feature sets are useful:

1. **Full model:** includes G1 and G2, giving the strongest predictive signal but relying on previous grades.
2. **Early-support model:** excludes G1 and G2, which is more useful for identifying students who may need support before those grades are available.

This distinction prevents the analysis from treating previous grades as if they were ordinary background factors.

In [ ]:
target = "G3"

# Keep all variables except the target.
X_full = df.drop(columns=[target])
y = df[target]

# Early-support features exclude previous-period grades.
X_early = X_full.drop(columns=["G1","G2"])

X_train_f, X_test_f, y_train, y_test = train_test_split(
    X_full, y, test_size=0.20, random_state=42
)
X_train_e, X_test_e, _, _ = train_test_split(
    X_early, y, test_size=0.20, random_state=42
)

def build_preprocessor(X):
    numeric = X.select_dtypes(include=np.number).columns.tolist()
    categorical = X.select_dtypes(exclude=np.number).columns.tolist()
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical)
    ])

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

def evaluate_models(X_train, X_test, y_train, y_test):
    rows, fitted = [], {}
    for name, model in models.items():
        pipe = Pipeline([("prep", build_preprocessor(X_train)), ("model", model)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rows.append({
            "Model": name,
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": mean_squared_error(y_test, pred, squared=False),
            "R2": r2_score(y_test, pred)
        })
        fitted[name] = (pipe, pred)
    return pd.DataFrame(rows).sort_values("RMSE"), fitted

full_results, full_models = evaluate_models(X_train_f, X_test_f, y_train, y_test)
early_results, early_models = evaluate_models(X_train_e, X_test_e, y_train, y_test)

print("Full-information models")
display(full_results.round(3))
print("Early-support models")
display(early_results.round(3))

## 5. Actual vs predicted and feature importance

In [ ]:
best_name = full_results.iloc[0]["Model"]
best_pipe, best_pred = full_models[best_name]

plt.figure(figsize=(7,7))
sns.scatterplot(x=y_test, y=best_pred)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
plt.plot(lims, lims, "--")
plt.xlabel("Actual G3")
plt.ylabel("Predicted G3")
plt.title(f"Actual vs Predicted – {best_name}")
plt.show()

# Feature importance for tree-based models when selected.
if best_name in ["Random Forest", "Gradient Boosting"]:
    prep = best_pipe.named_steps["prep"]
    model = best_pipe.named_steps["model"]
    feature_names = prep.get_feature_names_out()
    importance = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)
    display(importance.to_frame("Importance"))

    importance.sort_values().plot(kind="barh", figsize=(9,6), title="Top Feature Importances")
    plt.tight_layout()
    plt.show()

## 6. Conclusions
After running the notebook, report:

- Which study-time groups have the highest average final grades.
- Whether higher absences are associated with lower performance.
- How internet access and extracurricular activities relate to G3.
- How parental education differs across performance groups.
- Which model has the lowest MAE/RMSE and highest R².
- Whether the early-support model remains useful without G1/G2.
- The most influential features according to the selected model.

**Important dataset note:** UCI explicitly warns that G1 and G2 are strongly correlated with G3 because they are earlier grades from the same course. This is why the notebook reports both full-information and early-support models.